Question 39: 7 Day Streak
Difficulty: Medium
Link: https://www.interviewquery.com/questions/seven-day-streak?playlist=14-days-of-pandas

Problem Description:
===================
Given a table with event logs, find the percentage of users that had at least one seven-day streak
of visiting the same URL.  
  
_Note: Round the results to 2 decimal places. For example, if the result is 35.67% return 0.35._

**Example** :

**Input:**

`events` table

Column | Type  
---|---  
`user_id` | INTEGER  
`created_at` | DATETIME  
`url` | VARCHAR  
  
**Output**

Column | Type  
---|---  
`percent_of_users` | FLOAT


In [18]:
# Question 34: 7 Day Streak
# 
# Find users who have logged in for at least 7 consecutive days.

import pandas as pd

# Mock Data — 3 users with different streak patterns
logins_data = {
    'user_id': (
        [1]*10 +   # User 1: 10 consecutive days → has 7-day streak ✅
        [2]*8 +    # User 2: 6 consecutive, gap, then 2 → no 7-day streak ❌
        [3]*7 +    # User 3: exactly 7 consecutive → has streak ✅
        [4]*5      # User 4: only 5 consecutive → no streak ❌
    ),
    'login_date': pd.to_datetime(
        # User 1: Jan 1-10 (10 straight days)
        [f'2024-01-{d:02d}' for d in range(1, 11)] +
        # User 2: Jan 1-6 (6 days), then Jan 8-9 (gap on Jan 7)
        [f'2024-01-{d:02d}' for d in range(1, 7)] +
        ['2024-01-08', '2024-01-09'] +
        # User 3: Jan 1-7 (exactly 7)
        [f'2024-01-{d:02d}' for d in range(1, 8)] +
        # User 4: Jan 1-5 (only 5)
        [f'2024-01-{d:02d}' for d in range(1, 6)]
    )
}
logins = pd.DataFrame(logins_data)
# Expected: User 1 (10-day streak) and User 3 (7-day streak)
# User 2: max streak = 6 ❌, User 4: max streak = 5 ❌

def solution():
    # Write your solution here
    df = logins.copy()
    df['rank'] = logins.groupby('user_id')['login_date'].rank(method='first').astype(int)

    df['streak'] = df['login_date'] - pd.to_timedelta(df['rank'],unit='D')

    df = df.groupby('user_id')['streak'].size()

    answer = (df >= 5).sum() 

    display(answer)
    

solution()


np.int64(4)

In [19]:
# ✅ Optimal Solution

def solution_optimal():
    df = logins.copy()

    # 1. Deduplicate (in case of multiple logins same day)
    df = df.drop_duplicates(subset=["user_id", "login_date"])
    df = df.sort_values(["user_id", "login_date"]).reset_index(drop=True)

    # 2. Rank consecutive dates per user
    df["rank"] = df.groupby("user_id").cumcount()

    # # 3. Streak key: date - rank → same value for consecutive days
    df["streak_key"] = df["login_date"] - pd.to_timedelta(df["rank"], unit="D")

    # # 4. Group by (user_id, streak_key) to find each consecutive block
    streak_sizes = df.groupby(["user_id", "streak_key"]).size().reset_index(name="streak_len")


    # # 5. Find max streak per user
    max_streaks = streak_sizes.groupby("user_id")["streak_len"].max().reset_index()


    # # 6. Filter users with streak >= 7
    result = max_streaks[max_streaks["streak_len"] >= 7]
    
    return result

solution_optimal()


,user_id,streak_len
0,1,10
2,3,7
